In [1]:
# Проверка задания п2.1.4 на устройстве: трижды подряд спрашивает у платы адреса прошивки командой fw_info
# и записывает обмен в файл device-2-1-4.log.

import time
from datetime import datetime

import serial
from serial.tools import list_ports

VENDOR_ID = 0x2E8A
PRODUCT_ID = 0x000A

TASK = "2.1.4"
PROJECT = "211-command-usb"
LOG_NAME = "device-2-1-4.log"
COMMANDS = ["fw_info", "fw_info", "fw_info"]
ANSWER_S = 2


def find_board():
    for port in list_ports.comports():
        if port.vid == VENDOR_ID and port.pid == PRODUCT_ID:
            return port
    return None


def talk(board):
    exchange = []
    with serial.Serial(board.device, timeout=0.5) as port:
        time.sleep(0.2)
        port.reset_input_buffer()
        started = time.monotonic()
        for command in COMMANDS:
            port.write((command + "\n").encode("ascii"))
            exchange.append((time.monotonic() - started, "-->", command))
            deadline = time.monotonic() + ANSWER_S
            while time.monotonic() < deadline:
                line = port.readline().decode("ascii", "replace").strip()
                if line:
                    exchange.append((time.monotonic() - started, "<--", line))
                    print(line, end="\r\n")
    return exchange


def write_log(board, exchange):
    with open(LOG_NAME, "w", encoding="utf-8") as log:
        log.write("задание: " + TASK + "\n")
        log.write("проект: " + PROJECT + "\n")
        log.write("устройство: %04x:%04x\n" % (board.vid, board.pid))
        log.write("серийный номер: " + str(board.serial_number) + "\n")
        log.write("порт: " + board.device + "\n")
        log.write("начало: " + datetime.now().isoformat(timespec="seconds") + "\n")
        for moment, direction, text in exchange:
            log.write("%8.3f %s %s\n" % (moment, direction, text))
        answers = len(exchange) - len(COMMANDS)
        log.write("итог: отправлено команд %d, принято строк %d\n" % (len(COMMANDS), answers))


board = find_board()

if board is None:
    print("Плата не найдена. Проверьте кабель и запишите на плату прошивку задания.", end="\r\n")
else:
    print("Плата на порту " + board.device + ", отправляю команды", end="\r\n")
    exchange = talk(board)
    write_log(board, exchange)
    print("Обмен записан в " + LOG_NAME, end="\r\n")


Плата на порту COM3, отправляю команды
fw_info
dbg read_line:109 got fw_info
object           address    value
main             0x10000329 0xb5f0
fw_info          0x100005d5 0x4b3d
commands         0x10007a30
- enable         0x10007a34
- disable        0x10007a3c
- info           0x10007a44
- version        0x10007a4c
- ping           0x10007a54
- mem_info       0x10007a5c
- fw_info        0x10007a64
DEVICE_PROJECT   0x10007568 211-command-usb
DEVICE_BOARD     0x100075c0 pico
data_variable    0x20000f58 102
bss_variable     0x20001e40 2
stack_variable   0x20041f98 1946
heap_variable    0x20041f9c 1951
fw_info
dbg read_line:109 got fw_info
object           address    value
main             0x10000329 0xb5f0
fw_info          0x100005d5 0x4b3d
commands         0x10007a30
- enable         0x10007a34
- disable        0x10007a3c
- info           0x10007a44
- version        0x10007a4c
- ping           0x10007a54
- mem_info       0x10007a5c
- fw_info        0x10007a64
DEVICE_PROJECT   0x10007